In [32]:
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score, accuracy_score, recall_score, confusion_matrix
import numpy as np
import seaborn as sns

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", cache_folder='../../embedding_models').to(device)

In [34]:
df_1 = pd.read_csv("../../data/labeled_presence/label_Study_1_reviews.csv")

df_1_train = df_1.iloc[:100, :]

reviews_train = df_1_train["finalReview"].fillna("").tolist() 

In [35]:
review_embeddings_train = embed_model.encode(reviews_train, convert_to_tensor=False, show_progress_bar=True)

# df_1_train["embeddings"] = review_embeddings_train
# X = df_1_train["embeddings"]
X = review_embeddings_train

attribute_names = [
    'cleaning_service_quality',
    'order_packaging',
    'communication_and_responsiveness',
    'Driver_professionalism',
    'Service_speed'
]
y = df_1_train[attribute_names]

Batches: 100%|██████████| 4/4 [00:00<00:00, 62.48it/s]


In [50]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

# model_rf = RandomForestRegressor()
model_rf = RandomForestClassifier()
# model_xgb = XGBRegressor()
model_xgb = XGBClassifier()
model_lr = LogisticRegression()

model_rf.fit(X,y)
model_xgb.fit(X,y)
model_lr.fit(X,y.iloc[:,0])


LogisticRegression()

In [51]:
df_1_test = df_1.iloc[2502:, :]

reviews_test = df_1_test["finalReview"].fillna("").tolist() 

review_embeddings_test = embed_model.encode(reviews_test, convert_to_tensor=False, show_progress_bar=True)

# df_1_train["embeddings"] = review_embeddings_train
# X = df_1_train["embeddings"]
X = review_embeddings_test

attribute_names = [
    'cleaning_service_quality',
    'order_packaging',
    'communication_and_responsiveness',
    'Driver_professionalism',
    'Service_speed'
]
y_test = df_1_test[attribute_names]

y_pred_rf = model_rf.predict(X)
y_pred_xgb = model_xgb.predict(X)
y_pred_lr = model_lr.predict(X)

import numpy as np 

y_pred_binary_rf = (y_pred_rf > 0.5).astype(int)
y_pred_binary_xgb = (y_pred_xgb > 0.5).astype(int)
y_pred_binary_lr = (y_pred_lr > 0.5).astype(int)

Batches: 100%|██████████| 4/4 [00:00<00:00, 42.63it/s]


In [52]:
from sklearn.metrics import roc_auc_score

# y_test must be numpy array
y_true = y_test.values

# AUC per label
auc_per_label_rf = roc_auc_score(y_true, y_pred_rf, average=None)
auc_per_label_xgb = roc_auc_score(y_true, y_pred_xgb, average=None)

# Macro AUC (treat all labels equally)
auc_macro_rf = roc_auc_score(y_true, y_pred_rf, average="macro")
auc_macro_xgb = roc_auc_score(y_true, y_pred_xgb, average="macro")

# Micro AUC (treat all predictions equally)
auc_micro_rf = roc_auc_score(y_true, y_pred_rf, average="micro")
auc_micro_xgb = roc_auc_score(y_true, y_pred_xgb, average="micro")

print("RF")
print("AUC per label:", auc_per_label_rf)
print("Macro AUC:", auc_macro_rf)
print("Micro AUC:", auc_micro_rf)
print('\nXGB')
print("AUC per label:", auc_per_label_xgb)
print("Macro AUC:", auc_macro_xgb)
print("Micro AUC:", auc_micro_xgb)


RF
AUC per label: [0.46924603 0.48369565 0.46195652 0.45505618 0.48644226]
Macro AUC: 0.47127932963479074
Micro AUC: 0.5079485451072914

XGB
AUC per label: [0.46924603 0.48369565 0.46195652 0.45505618 0.48644226]
Macro AUC: 0.47127932963479074
Micro AUC: 0.5079485451072914


In [53]:
from sklearn.metrics import confusion_matrix
import pandas as pd

attribute_names = [
    'cleaning_service_quality',
    'order_packaging',
    'communication_and_responsiveness',
    'Driver_professionalism',
    'Service_speed'
]

# Convert to numpy arrays
y_true = y_test.values
y_pred_bin_rf = y_pred_binary_rf
y_pred_bin_xgb = y_pred_binary_xgb
y_pred_bin_lr = y_pred_binary_lr

print("FOR RF")
conf_matrices_rf = {}

for i, attr in enumerate(attribute_names):
    cm = confusion_matrix(y_true[:, i], y_pred_bin_rf[:, i])
    conf_matrices_rf[attr] = cm
    print(f"\nConfusion Matrix for {attr}:\n{cm}")

print("FOR XGB")
conf_matrices_xgb = {}

for i, attr in enumerate(attribute_names):
    cm = confusion_matrix(y_true[:, i], y_pred_bin_xgb[:, i])
    conf_matrices_xgb[attr] = cm
    print(f"\nConfusion Matrix for {attr}:\n{cm}")

print("FOR LOGISTIC REGRESSION")
cm = confusion_matrix(y_true[:, 0], y_pred_bin_lr)
print(cm)


FOR RF

Confusion Matrix for cleaning_service_quality:
[[47 25]
 [20  8]]

Confusion Matrix for order_packaging:
[[89  3]
 [ 8  0]]

Confusion Matrix for communication_and_responsiveness:
[[85  7]
 [ 8  0]]

Confusion Matrix for Driver_professionalism:
[[81  8]
 [11  0]]

Confusion Matrix for Service_speed:
[[56 13]
 [26  5]]
FOR XGB

Confusion Matrix for cleaning_service_quality:
[[47 25]
 [20  8]]

Confusion Matrix for order_packaging:
[[89  3]
 [ 8  0]]

Confusion Matrix for communication_and_responsiveness:
[[85  7]
 [ 8  0]]

Confusion Matrix for Driver_professionalism:
[[81  8]
 [11  0]]

Confusion Matrix for Service_speed:
[[56 13]
 [26  5]]
FOR LOGISTIC REGRESSION
[[72  0]
 [28  0]]
